In [2]:
# ============================================================================
# 02_perturbation.ipynb  —  Controlled perturbations P1-P5
# ----------------------------------------------------------------------------
# Single-cell notebook. All code and comments are in English.
# Figures: seaborn grayscale, no captions, dpi=600, saved as both PNG and PDF.
# Run order: 00 -> 01 -> 02 -> 03 -> 04 -> 05.
# ============================================================================
import sys, os, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd

# Make src/ importable regardless of the notebook's working directory.
_HERE = Path.cwd()
_ROOT = _HERE if (_HERE / "src").exists() else _HERE.parent
sys.path.insert(0, str(_ROOT / "src"))
from common import (PROJECT_ROOT, DATA_DIR, RESULTS_DIR, FIGURES_DIR, TABLES_DIR,
                    load_env, set_grayscale_style, save_fig, save_table,
                    require_input, RANDOM_SEED)

import matplotlib.pyplot as plt
import seaborn as sns
set_grayscale_style()
np.random.seed(RANDOM_SEED)
warnings.filterwarnings("ignore")
# NOTE: absolute local paths are never printed (anonymity for double-blind review).

# ----------------------------------------------------------------------------
# PURPOSE
#   Derive controlled variants from each generated (rating, rationale), each
#   isolating ONE defect while holding surface form otherwise fixed:
#     P1 rating-rationale inconsistency : shift the grade by k notches, keep text
#     P2 remove qualitative justification: delete the support clause(s)
#     P3 factual error                   : inject a claim contradicting the input
#     P4 direction flip                  : reverse the support polarity
#     P5 verbosity control               : pad with repetition; content unchanged
#   A sound judge should drop on P1-P4 and stay robust to P5; the technical
#   metric should move mainly for P3. P1 is applied at k=1,2,3 (dose-response).
#
# INPUT   data/generated.parquet
# OUTPUT  data/perturbed.parquet  (case_id, condition, pert, k, rating, rationale)
# ----------------------------------------------------------------------------
cfg = load_env()
from openai import OpenAI
client = OpenAI(api_key=cfg["api_key"])
EDIT_MODEL = cfg["mid"]

gen = pd.read_parquet(require_input(DATA_DIR / "generated.parquet",
                                    hint="Run 01_generation first."))
RATING_ORDER = ["D","C","CC","CCC","B","BB","BBB","A","AA","AAA"]
idx = {g:i for i,g in enumerate(RATING_ORDER)}

def shift_grade(g, k):
    if g not in idx: return g
    return RATING_ORDER[max(0, min(len(RATING_ORDER)-1, idx[g]+k))]

def llm_edit(instruction, rationale):
    r = client.chat.completions.create(
        model=EDIT_MODEL, temperature=0.0,
        messages=[{"role":"system","content":
                   "You edit a credit rationale as instructed. Return only the edited text."},
                  {"role":"user","content": instruction + "\n\nRATIONALE:\n" + str(rationale)}])
    return r.choices[0].message.content.strip()

rows = []
for _, r in gen.iterrows():
    base = {"case_id": r.case_id, "cik": r.cik, "rating_year": r.rating_year,
            "delta": r.delta, "condition": r.condition}
    g, t = r.rating_gen, r.rationale
    # P0 original
    rows.append({**base, "pert":"P0", "k":0, "rating":g, "rationale":t})
    # P1 dose-response k=1..3
    for k in (1,2,3):
        rows.append({**base, "pert":"P1", "k":k, "rating":shift_grade(g,k), "rationale":t})
    # P2 remove qualitative justification
    rows.append({**base, "pert":"P2", "k":0, "rating":g,
                 "rationale": llm_edit("Delete every sentence that provides a "
                 "qualitative support justification (parent/group/government "
                 "support, industry, governance). Keep the rest verbatim.", t)})
    # P3 inject a factual error
    rows.append({**base, "pert":"P3", "k":0, "rating":g,
                 "rationale": llm_edit("Insert exactly one quantitative claim that "
                 "contradicts the financials (e.g. flip a ratio's direction). "
                 "Change nothing else.", t)})
    # P4 flip support direction
    rows.append({**base, "pert":"P4", "k":0, "rating":g,
                 "rationale": llm_edit("Reverse the polarity of the qualitative "
                 "support argument (support -> drag, or drag -> support) while "
                 "keeping the same grade and surface form.", t)})
    # P5 verbosity padding (content unchanged)
    rows.append({**base, "pert":"P5", "k":0, "rating":g,
                 "rationale": (str(t) + " " + str(t))})

pert = pd.DataFrame(rows)
pert.to_parquet(DATA_DIR / "perturbed.parquet", index=False)
print(pert.groupby("pert").size())


pert
P0     40
P1    120
P2     40
P3     40
P4     40
P5     40
dtype: int64
